# Qwen3 VL AutoTagger CLI - Production Colab Runner

Use this notebook for a clean production run on Colab (T4 or better).

## What this notebook does
- Clones or updates the repository
- Installs Python dependencies and `exiftool`
- Uploads images for batch tagging
- Runs CLI with XMP writing enabled by default
- Verifies JSONL output and previews XMP metadata
- Downloads all outputs as a ZIP archive

## Runtime
Set Runtime to GPU in Colab: `Runtime -> Change runtime type -> T4 GPU`.


In [ ]:
from pathlib import Path
import getpass
import json
import os
import shutil
import subprocess
import sys

from google.colab import files

REPO_URL = 'https://github.com/ekkonwork/qwen3-vl-autotagger-cli.git'
WORKDIR = Path('/content/qwen3-vl-autotagger-cli')
INPUT_DIR = Path('/content/input_images')
OUTPUT_DIR = Path('/content/outputs')

def run(cmd, cwd=None, check=True, redact=None):
    cmd = [str(x) for x in cmd]
    shown = ' '.join(cmd)
    for s in (redact or []):
        if s:
            shown = shown.replace(s, '***')
    print('$', shown)
    res = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if res.stdout:
        print(res.stdout)
    if res.stderr:
        print(res.stderr)
    if check and res.returncode != 0:
        raise RuntimeError(f'Command failed with code {res.returncode}: {shown}')
    return res

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if WORKDIR.exists():
    print(f'Repo exists: {WORKDIR}')
    run(['git', '-C', str(WORKDIR), 'status'], check=False)
    run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'], check=False)
else:
    token = os.environ.get('GITHUB_TOKEN', '').strip()
    if not token:
        token = getpass.getpass('GitHub token for private repo (leave empty if public): ').strip()

    clone_url = REPO_URL
    if token and REPO_URL.startswith('https://github.com/'):
        clone_url = REPO_URL.replace('https://', f'https://x-access-token:{token}@', 1)

    run(['git', 'clone', clone_url, str(WORKDIR)], redact=[token])

print('Setup paths ready.')


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel'])
run([sys.executable, '-m', 'pip', 'install', '-r', str(WORKDIR / 'requirements.txt')])
run([sys.executable, str(WORKDIR / 'install.py')], cwd=str(WORKDIR), check=False)
run(['exiftool', '-ver'], check=False)
print('Dependencies installed.')


In [ ]:
print('Upload input images (1..N files):')
uploaded = files.upload()
saved = []
for name, data in uploaded.items():
    p = INPUT_DIR / name
    with p.open('wb') as f:
        f.write(data)
    saved.append(p)

if not saved:
    raise RuntimeError('No files uploaded.')

print(f'Saved {len(saved)} files to {INPUT_DIR}')


In [ ]:
cli_cmd = [
    sys.executable, '-m', 'qwen3_vl_autotagger_cli.cli',
    str(INPUT_DIR),
    '--recursive',
    '--output-dir', str(OUTPUT_DIR),
    '--output-format', 'jpg',
    '--file-prefix', 'colab',
    '--metadata-jsonl', str(OUTPUT_DIR / 'metadata.jsonl'),
    '--write-xmp',
    '--require-exiftool',
    '--log-tags',
    '--attempts', '2',
    '--max-keywords', '50',
    '--max-new-tokens', '600',
    '--temperature', '0.7',
    '--top-p', '0.9',
    '--repetition-penalty', '1.15',
    '--min-pixels', str(256 * 28 * 28),
    '--max-pixels', str(756 * 756),
    '--allow-resize',
]

run(cli_cmd, cwd=str(WORKDIR))
print('CLI run finished.')


In [ ]:
image_outputs = sorted([p for p in OUTPUT_DIR.glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}])
metadata_path = OUTPUT_DIR / 'metadata.jsonl'

print('Image outputs:')
for p in image_outputs:
    print('-', p)

if not metadata_path.exists():
    raise RuntimeError('metadata.jsonl not found')

records = []
for line in metadata_path.read_text(encoding='utf-8', errors='ignore').splitlines():
    line = line.strip()
    if not line:
        continue
    records.append(json.loads(line))

print(f'Metadata records: {len(records)}')
for idx, r in enumerate(records, 1):
    title = r.get('title', '')
    kw = r.get('keywords', [])
    print(f'[{idx}] title_len={len(title)} keywords={len(kw)} output={r.get("output_path", "")}')

if image_outputs:
    first = str(image_outputs[0])
    print('\nXMP preview for first output:')
    run(['exiftool', '-XMP-dc:Title', '-XMP-dc:Description', '-XMP-dc:Subject', first], check=False)

print('Verification complete.')


In [ ]:
archive = shutil.make_archive('/content/qwen3_vl_autotagger_outputs', 'zip', root_dir=str(OUTPUT_DIR))
print('Archive ready:', archive)
files.download(archive)
